In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib   as mpl

In [ ]:
def function_sub(x, a):
    return (1. + x**2. / 2.) / np.sqrt(1 + a * x**2. / 2.)

def function_main(x, x_0, a):
    return function_sub(x-x_0, a)# - function_sub(x, a)

In [ ]:
x_array = np.logspace(-3, 2, 1000)

a_T = 1.5

x_w = 1E-1

y_scale = 1E0

y_w_array = function_main(x_array * y_scale, x_w * y_scale, a_T)
y_i_array = function_main(x_array * y_scale, 0. * y_scale, a_T)

print(y_w_array)

print(y_i_array)

In [ ]:
mpl.rcParams['font.size'] = 15

fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(2, 1)
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0])

ax0.loglog(x_array, y_w_array, c='blue')
ax0.loglog(x_array, y_i_array, c='red')
ax1.loglog(x_array, np.abs(y_w_array - y_i_array) / y_i_array, c='k')

ax0.minorticks_on()
ax0.grid(which='both', alpha=0.5)
ax1.minorticks_on()
ax1.grid(which='both', alpha=0.5)

fig.tight_layout()

fig.show()

In [ ]:
# Solve z(ζ) from the implicit equation in the image and plot.
# Equation interpreted as:
# h(z, ζ) = ( 1 - R(z, ζ) )^2 - 1/B^2 = 0,
# R(z, ζ) = ((1 - z)^2 * ζ^2 + 2)/(ζ^2 + 2) * sqrt( (A*ζ^2 + 2)/(A*(1 - z)^2 * ζ^2 + 2) )
#
# All parameters positive. ζ>0, z>0.
#
# The code solves (1 - R) - s/B = 0 for s=±1 using Brent's method,
# for each ζ over a user-set range. Feel free to edit A, B, z bounds, and ζ grid.

import numpy as np
import matplotlib.pyplot as plt
from math import sqrt, isfinite
from scipy.optimize import brentq

# ---- parameters you may edit ----
A = 1.5
B = 10.0
z_min, z_max = 1e-9, 100.
zeta_min, zeta_max, n_zeta = 0.1, 100.0, 400

# ---- core functions ----
def R_of(z, zeta, A):
    num = (1.0 - z)**2 * zeta**2 + 2.0
    den = zeta**2 + 2.0
    inner = (A*zeta**2 + 2.0) / (A*(1.0 - z)**2 * zeta**2 + 2.0)
    return (num/den) * sqrt(inner)

def g_of(z, zeta, A, B, s):
    # root of g(z) = 0 gives solution for a given sign branch s ∈ {+1, -1}
    return (1.0 - R_of(z, zeta, A)) - (s / B)

# ---- solver per ζ ----
def solve_z_for_zeta(zeta, A, B, s):
    # Try multiple brackets across [z_min, z_max] to find a sign change.
    # Coarse grid to locate sign flips.
    grid = np.linspace(z_min, z_max, 2001)
    vals = []
    for z in grid:
        try:
            vals.append(g_of(z, zeta, A, B, s))
        except ValueError:
            vals.append(np.nan)
    vals = np.array(vals)

    # Find intervals where sign changes and values are finite
    roots = []
    for i in range(len(grid)-1):
        a, b = grid[i], grid[i+1]
        fa, fb = vals[i], vals[i+1]
        if not (isfinite(fa) and isfinite(fb)):
            continue
        if fa == 0.0:
            roots.append(a)
            continue
        if fa*fb < 0.0:
            try:
                root = brentq(lambda zz: g_of(zz, zeta, A, B, s), a, b, maxiter=200)
                roots.append(root)
            except ValueError:
                pass
    # Return smallest positive root if multiple
    if roots:
        return min(roots)
    return np.nan

# ---- sweep over ζ ----
zetas = np.logspace(np.log10(zeta_min), np.log10(zeta_max), n_zeta)
z_branch_plus  = np.array([solve_z_for_zeta(ze, A, B, s=+1) for ze in zetas])
z_branch_minus = np.array([solve_z_for_zeta(ze, A, B, s=-1) for ze in zetas])

# ---- plot ----
plt.figure(figsize=(7,5))
mask_p = np.isfinite(z_branch_plus)
mask_m = np.isfinite(z_branch_minus)
plt.plot(zetas[mask_p], z_branch_plus[mask_p], label="branch: 1 - R = +1/B")
plt.plot(zetas[mask_m], z_branch_minus[mask_m], label="branch: 1 - R = -1/B")
plt.xscale('log')
plt.xlabel(r"$\zeta$")
plt.ylabel(r"$z(\zeta)$")
plt.xlim(zeta_min, zeta_max)
plt.title(rf"Implicit solution for $A={A}$, $B={B}$")
plt.legend()
plt.grid(True, which="both")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# ---------------- core ----------------
def R_of(z, zeta, A):
    z, zeta = np.asarray(z), np.asarray(zeta)
    num = (1.0 - z)**2 * zeta**2 + 2.0
    den = zeta**2 + 2.0
    inner = (A*zeta**2 + 2.0) / (A*(1.0 - z)**2 * zeta**2 + 2.0)
    return (den/num) * np.sqrt(1./inner)

def h_of(z, zeta, A, B):
    return (1.0 - R_of(z, zeta, A))**2 - 1.0/(B**2)

def g_of(z, zeta, A, B, s):
    return (1.0 - R_of(z, zeta, A)) - (s / B)

def solve_all_roots_for_zeta(zeta, A, B, s, z_min, z_max, n_grid=2001, logspace=True):
    """同一ζでの全実根を返す"""
    grid = (np.logspace(np.log10(z_min), np.log10(z_max), n_grid)
            if logspace else np.linspace(z_min, z_max, n_grid))
    vals = np.array([g_of(z, zeta, A, B, s) for z in grid])
    roots = []
    for a, b, fa, fb in zip(grid[:-1], grid[1:], vals[:-1], vals[1:]):
        if not (np.isfinite(fa) and np.isfinite(fb)):
            continue
        if fa == 0.0:
            roots.append(a); continue
        if fa*fb < 0.0:
            roots.append(brentq(lambda zz: g_of(zz, zeta, A, B, s), a, b, maxiter=200))
    return roots

# ---------------- plot ----------------
def plot_h_region_and_branches(
    A=1.5, B=3.0,
    z_min=5e-2, z_max=20.0, z_N=800,
    zeta_min=0.1, zeta_max=100.0, zeta_N=400,
    eps=0.1, U=0.1, M = 1.,
    log_axes=True
    ):
    # grids
    z_grid    = np.logspace(np.log10(z_min),   np.log10(z_max),   z_N)
    zeta_grid = np.logspace(np.log10(zeta_min),np.log10(zeta_max),zeta_N)
    ZETA, Z = np.meshgrid(zeta_grid, z_grid)

    # 条件1: h<0
    H = h_of(Z, ZETA, A, B)
    cond1 = (H < 0.0)

    # 条件2: (2 z^2 ζ^2)/(2 + A(1-z)^2 ζ^2) < eps^2 U^2
    lhs = (2.0*Z**2 * ZETA**2) / (2.0 + A*(1.0 - Z)**2 * ZETA**2)
    rhs = (eps**2)*(U**2)
    cond2 = (lhs < rhs)

    # 両方
    cond_both = cond1 & cond2

    # カテゴリマップ: 0=なし, 1=灰, 2=橙, 3=赤
    regions = np.zeros_like(H, dtype=np.uint8)
    regions[cond1]    = 1
    regions[cond2]    = 2
    regions[cond_both]= 3

    # 描画
    fig, ax = plt.subplots(figsize=(7,5), dpi=200)

    cmap = ListedColormap([
        (0,0,0,0.0),        # none
        (0.60,0.60,0.60,0.35),  # gray
        (1.00,0.80,0.00,0.35),  # orange
        (1.00,0.00,0.00,0.35)   # red
    ])
    ax.pcolormesh(zeta_grid, z_grid, regions, shading="auto", cmap=cmap, zorder=0)

    # 参考: 境界線
    ax.contour(zeta_grid, z_grid, H, levels=[0.0], colors='k', linewidths=1.0, zorder=1)
    ax.contour(zeta_grid, z_grid, lhs, levels=[rhs], colors='orange', linewidths=1.0, linestyles='--', zorder=1)

    # 分枝（±）
    pts = []
    for s in (+1, -1):
        for ze in zeta_grid:
            for r in solve_all_roots_for_zeta(ze, A, B, s, z_min, z_max, n_grid=2001, logspace=True):
                pts.append((ze, r, s))
    if pts:
        P = np.array(pts)
        for s, lab in [(+1, '1−R=+1/B'), (-1, '1−R=−1/B')]:
            m = P[:,2] == s
            ax.plot(P[m,0], P[m,1], '.', ms=1.2, label=lab, zorder=2)

    ax.set_xlabel(r'$\zeta = f_{\mathrm{sc}} v_{\mathrm{thi}} /( f_{\mathrm{ci}} V_{\mathrm{sys}\perp} \cos\theta )$')
    ax.set_ylabel(r'$z = f_{\mathrm{w}}/f_{\mathrm{sc}}$')
    if log_axes:
        ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(zeta_min, zeta_max); ax.set_ylim(z_min, z_max)
    ax.set_title(r"$T_{\mathrm{e}}/T_{\mathrm{i}}$="
                 f"{(A-1.):.2f}, " + r"$\Delta$=" + f"{B}, "
                 + r"$\varepsilon$=" + f"{eps:.2f}, " + r"$U$=" + f"{U:.2f}, " + r"$m_{\mathrm{i}} / m_{\mathrm{H}^{+}}$=" + f"{M:.2f}")
    # 凡例
    handles = [
        mpatches.Patch(facecolor=(0.60,0.60,0.60,0.35), edgecolor='none', label=r'$h<0$'),
        mpatches.Patch(facecolor=(1.00,0.80,0.00,0.35), edgecolor='none', label=r'$\frac{2 z^2 \zeta^2}{2+A(1-z)^2\zeta^2} < \varepsilon^2 U^2$'),
        mpatches.Patch(facecolor=(1.00,0.00,0.00,0.35), edgecolor='none', label='both')
    ]
    ax.plot(zeta_grid, M * U / 10. / zeta_grid, lw=4.0, c='purple', label=r'$z=\frac{MU}{10 \zeta}$')
    ax.legend(handles=handles, loc='best')
    ax.minorticks_on(); ax.grid(which='both', alpha=0.5)
    plt.tight_layout()
    return fig, ax, (zeta_grid, z_grid, regions)

# 例
if __name__ == "__main__":
    plot_h_region_and_branches(A=1.5, B=10.0, eps=1E-1, U=8., M=1)
    plt.show()